# Notebook 4 — New Frappuccino Design Optimizer

**Algorithm 1:** Given macro conditions (season, temperature, CPI/wage trends), what product attributes maximize predicted sales?

**Approach:**
1. Analyze historical (synthetic) purchase patterns to extract what drives Frappuccino selection
2. Build a scoring function: `product_score = f(calories, sugar, caffeine, price, temp, season, CPI)`
3. Use constrained optimization to find the optimal new product design
4. Generate actionable product specifications with predicted revenue impact

**Key constraint:** All optimization inputs are grounded in real menu data and real macro indicators. The synthetic transactions provide the behavioral bridge.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.optimize import minimize
from pathlib import Path
from IPython.display import display

np.random.seed(42)

# ── Paths ────────────────────────────────────────────────────────────
ON_KAGGLE = Path('/kaggle/working').exists()
if ON_KAGGLE:
    DATA_DIR = Path('/kaggle/input/starbucks-recommendation-engine')
else:
    DATA_DIR = Path('../data')

menu = pd.read_csv((DATA_DIR / 'raw' if not ON_KAGGLE else DATA_DIR) / 'starbucks_menu.csv')
fred = pd.read_csv((DATA_DIR / 'raw' if not ON_KAGGLE else DATA_DIR) / 'fred_macro.csv', parse_dates=['date'])
txn = pd.read_csv((DATA_DIR / 'processed' if not ON_KAGGLE else DATA_DIR) / 'synthetic_transactions.csv', parse_dates=['date'])

print(f'Menu:         {menu.shape}')
print(f'Transactions: {txn.shape}')
print(f'FRED:         {fred.shape}')

## Section 1 — Frappuccino Market Analysis

First, understand the current Frappuccino landscape: which attributes drive selection?

In [ ]:
# ── Filter to Frappuccino transactions ────────────────────────────────
frapp_cats = ['Frappuccino® Blended Coffee', 'Frappuccino® Blended Crème',
              'Frappuccino® Light Blended Coffee']
txn_frapp = txn[txn['category'].isin(frapp_cats)].copy()
print(f'Frappuccino transactions: {len(txn_frapp):,} ({len(txn_frapp)/len(txn):.1%} of total)')

# ── Existing Frappuccino menu profile ─────────────────────────────────
frapp_menu = menu[menu['category'].isin(frapp_cats)].copy()
print(f'\nFrappuccino menu items: {len(frapp_menu)}')

fig, axes = plt.subplots(2, 2, figsize=(12, 8))

# Calorie distribution
axes[0, 0].hist(frapp_menu['calories'], bins=15, color='#00704A', edgecolor='white')
axes[0, 0].set_title('Frappuccino Calorie Distribution')
axes[0, 0].set_xlabel('Calories')
axes[0, 0].axvline(frapp_menu['calories'].median(), color='red', linestyle='--', label=f'Median: {frapp_menu["calories"].median():.0f}')
axes[0, 0].legend()

# Sugar distribution
axes[0, 1].hist(frapp_menu['sugar_g'], bins=15, color='#E76F51', edgecolor='white')
axes[0, 1].set_title('Frappuccino Sugar Distribution')
axes[0, 1].set_xlabel('Sugar (g)')
axes[0, 1].axvline(frapp_menu['sugar_g'].median(), color='red', linestyle='--', label=f'Median: {frapp_menu["sugar_g"].median():.0f}g')
axes[0, 1].legend()

# Price distribution
axes[1, 0].hist(frapp_menu['price_usd'], bins=15, color='#457B9D', edgecolor='white')
axes[1, 0].set_title('Frappuccino Price Distribution')
axes[1, 0].set_xlabel('Price ($)')
axes[1, 0].axvline(frapp_menu['price_usd'].median(), color='red', linestyle='--', label=f'Median: ${frapp_menu["price_usd"].median():.2f}')
axes[1, 0].legend()

# Caffeine vs Calories scatter
axes[1, 1].scatter(frapp_menu['caffeine_mg'], frapp_menu['calories'], 
                   c=frapp_menu['price_usd'], cmap='YlOrRd', s=60, edgecolor='gray', alpha=0.8)
axes[1, 1].set_title('Caffeine vs Calories (color=price)')
axes[1, 1].set_xlabel('Caffeine (mg)')
axes[1, 1].set_ylabel('Calories')

plt.suptitle('Current Frappuccino Menu Profile', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print('\n=== Frappuccino Menu Stats ===')
for col in ['calories', 'sugar_g', 'caffeine_mg', 'price_usd']:
    print(f'  {col:15s}: mean={frapp_menu[col].mean():.1f}, min={frapp_menu[col].min():.1f}, max={frapp_menu[col].max():.1f}')

## Section 2 — Demand Drivers Analysis

What external factors drive Frappuccino demand?

In [ ]:
# ── Temperature effect on Frappuccino share ──────────────────────────
txn['is_frapp'] = txn['category'].isin(frapp_cats).astype(int)
txn['temp_bin'] = pd.cut(txn['temp_f'], bins=10)

temp_frapp = txn.groupby('temp_bin', observed=True).agg(
    frapp_share=('is_frapp', 'mean'),
    avg_price=('total_price', 'mean'),
    count=('transaction_id', 'size')
).reset_index()

# ── Monthly seasonality ──────────────────────────────────────────────
txn['month'] = txn['date'].dt.month
month_frapp = txn.groupby('month').agg(
    frapp_share=('is_frapp', 'mean'),
    avg_frapp_price=('total_price', lambda x: x[txn.loc[x.index, 'is_frapp']==1].mean() if txn.loc[x.index, 'is_frapp'].sum()>0 else 0),
    total_txn=('transaction_id', 'size')
).reset_index()

# ── CPI effect ───────────────────────────────────────────────────────
txn['cpi_bin'] = pd.qcut(txn['cpi'], q=5, duplicates='drop')
cpi_effect = txn.groupby('cpi_bin', observed=True).agg(
    avg_price=('total_price', 'mean'),
    avg_upcharge=('upcharge', 'mean'),
    custom_rate=('n_customizations', lambda x: (x > 0).mean()),
).reset_index()

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Temperature vs Frapp share
axes[0].bar(range(len(temp_frapp)), temp_frapp['frapp_share'], color='#00704A', edgecolor='white')
axes[0].set_title('Temperature → Frappuccino Share')
axes[0].set_xlabel('Temperature Bin (cold → hot)')
axes[0].set_ylabel('Frappuccino Share')
axes[0].set_xticks(range(len(temp_frapp)))
axes[0].set_xticklabels([f'{x.left:.0f}-{x.right:.0f}°F' for x in temp_frapp['temp_bin']], rotation=45, fontsize=7)

# Monthly seasonality
axes[1].bar(month_frapp['month'], month_frapp['frapp_share'], color='#E76F51', edgecolor='white')
axes[1].set_title('Monthly Frappuccino Share')
axes[1].set_xlabel('Month')
axes[1].set_ylabel('Frappuccino Share')
axes[1].set_xticks(range(1, 13))
axes[1].set_xticklabels(['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec'], fontsize=8)

# CPI vs behavior
axes[2].bar(range(len(cpi_effect)), cpi_effect['avg_price'], color='#457B9D', edgecolor='white', label='Avg Price')
axes[2].set_title('CPI Level → Average Price Paid')
axes[2].set_xlabel('CPI Quintile (low → high)')
axes[2].set_ylabel('Average Price ($)')

plt.suptitle('Demand Drivers for Frappuccino', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## Section 3 — Product Scoring Function

We define a scoring function that predicts relative demand for a hypothetical new Frappuccino based on its attributes and market conditions.

**Score = Desirability × Affordability × Seasonal Fit**

Where:
- **Desirability** = how well the product fits consumer preferences (calorie sweet spot, caffeine level)
- **Affordability** = price sensitivity adjusted by CPI/wage trends (higher CPI → more price-sensitive)
- **Seasonal Fit** = temperature and month alignment (Frappuccinos peak in summer)

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# PRODUCT SCORING FUNCTION
# ══════════════════════════════════════════════════════════════════════

# Derived from synthetic data analysis
FRAPP_IDEAL_CALORIES = frapp_menu['calories'].median()  # Sweet spot
FRAPP_IDEAL_SUGAR = frapp_menu['sugar_g'].median()
FRAPP_PRICE_MEAN = frapp_menu['price_usd'].mean()
FRAPP_PRICE_STD = frapp_menu['price_usd'].std()

# Latest macro conditions
latest_cpi = fred['cpi'].iloc[-1]
latest_wage = fred['avg_hourly_earnings'].iloc[-1]
latte_index = latest_wage / FRAPP_PRICE_MEAN  # How many frappuccinos per hour of work

print(f'Ideal calories: {FRAPP_IDEAL_CALORIES:.0f}')
print(f'Ideal sugar:    {FRAPP_IDEAL_SUGAR:.0f}g')
print(f'Price mean/std: ${FRAPP_PRICE_MEAN:.2f} ± ${FRAPP_PRICE_STD:.2f}')
print(f'Latest CPI:     {latest_cpi:.1f}')
print(f'Latest wage:    ${latest_wage:.2f}/hr')
print(f'Latte index:    {latte_index:.1f} drinks/hr wage')


def product_score(calories, sugar_g, caffeine_mg, price, temp_f, month, cpi, wage):
    """
    Score a hypothetical Frappuccino product.
    Higher score = higher predicted relative demand.
    
    All parameters are transparent and inspectable.
    """
    # --- Desirability (0-1) ---
    # Calorie: bell curve around ideal, penalize extremes
    cal_score = np.exp(-0.5 * ((calories - FRAPP_IDEAL_CALORIES) / 80) ** 2)
    
    # Sugar: slight preference for moderate sweetness
    sugar_score = np.exp(-0.5 * ((sugar_g - FRAPP_IDEAL_SUGAR) / 15) ** 2)
    
    # Caffeine: flexible, slight bonus for moderate caffeine
    caff_score = 0.7 + 0.3 * np.exp(-0.5 * ((caffeine_mg - 100) / 80) ** 2)
    
    desirability = (cal_score * 0.4 + sugar_score * 0.3 + caff_score * 0.3)
    
    # --- Affordability (0-1) ---
    # Price sensitivity increases with CPI
    cpi_factor = latest_cpi / cpi  # > 1 when CPI was lower (less sensitive)
    price_z = (price - FRAPP_PRICE_MEAN) / FRAPP_PRICE_STD
    affordability = 1.0 / (1.0 + np.exp(price_z * cpi_factor))  # Sigmoid
    
    # Wage adjustment: higher wages reduce price sensitivity
    wage_factor = wage / latest_wage
    affordability = affordability ** (1.0 / max(wage_factor, 0.5))
    
    # --- Seasonal Fit (0-1) ---
    # Temperature: Frappuccinos peak above 70°F
    temp_score = 1.0 / (1.0 + np.exp(-(temp_f - 65) / 10))
    
    # Month: summer bonus
    summer_months = {5: 0.8, 6: 1.0, 7: 1.0, 8: 1.0, 9: 0.8}
    month_bonus = summer_months.get(month, 0.4)
    
    seasonal = temp_score * 0.6 + month_bonus * 0.4
    
    # --- Combined Score ---
    score = desirability * 0.35 + affordability * 0.35 + seasonal * 0.30
    
    return score, {
        'desirability': desirability,
        'cal_score': cal_score,
        'sugar_score': sugar_score,
        'caff_score': caff_score,
        'affordability': affordability,
        'seasonal': seasonal,
        'temp_score': temp_score,
    }

# Test with current top frappuccino
test_score, test_detail = product_score(
    calories=300, sugar_g=45, caffeine_mg=90, price=5.25,
    temp_f=80, month=7, cpi=latest_cpi, wage=latest_wage
)
print(f'\nTest product score: {test_score:.4f}')
for k, v in test_detail.items():
    print(f'  {k}: {v:.4f}')

## Section 4 — Constrained Optimization

Find the optimal product attributes for different market scenarios.

**Constraints:**
- Calories: 150-500 (realistic Frappuccino range)
- Sugar: 20-65g
- Caffeine: 0-200mg
- Price: $4.00-$7.50

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# CONSTRAINED OPTIMIZATION
# ══════════════════════════════════════════════════════════════════════

def optimize_product(temp_f, month, cpi, wage, label=''):
    """Find optimal product attributes given market conditions."""
    def neg_score(x):
        s, _ = product_score(x[0], x[1], x[2], x[3], temp_f, month, cpi, wage)
        return -s
    
    bounds = [(150, 500), (20, 65), (0, 200), (4.00, 7.50)]
    
    # Multi-start to avoid local optima
    best_result = None
    best_score = -np.inf
    for _ in range(20):
        x0 = [np.random.uniform(b[0], b[1]) for b in bounds]
        result = minimize(neg_score, x0, method='L-BFGS-B', bounds=bounds)
        if -result.fun > best_score:
            best_score = -result.fun
            best_result = result
    
    cal, sug, caf, price = best_result.x
    score, detail = product_score(cal, sug, caf, price, temp_f, month, cpi, wage)
    
    return {
        'scenario': label,
        'calories': round(cal),
        'sugar_g': round(sug),
        'caffeine_mg': round(caf),
        'price': round(price, 2),
        'score': round(score, 4),
        'desirability': round(detail['desirability'], 4),
        'affordability': round(detail['affordability'], 4),
        'seasonal': round(detail['seasonal'], 4),
    }

# ── Run optimization across scenarios ────────────────────────────────
scenarios = [
    {'temp_f': 85, 'month': 7, 'cpi': latest_cpi, 'wage': latest_wage,
     'label': 'Summer Peak (July, 85°F)'},
    {'temp_f': 40, 'month': 1, 'cpi': latest_cpi, 'wage': latest_wage,
     'label': 'Winter (January, 40°F)'},
    {'temp_f': 70, 'month': 4, 'cpi': latest_cpi, 'wage': latest_wage,
     'label': 'Spring Shoulder (April, 70°F)'},
    {'temp_f': 85, 'month': 7, 'cpi': latest_cpi * 1.05, 'wage': latest_wage,
     'label': 'Summer + High Inflation (+5% CPI)'},
    {'temp_f': 85, 'month': 7, 'cpi': latest_cpi, 'wage': latest_wage * 1.10,
     'label': 'Summer + Wage Growth (+10%)'},
]

results = []
for s in scenarios:
    r = optimize_product(**s)
    results.append(r)
    print(f"\n{r['scenario']}:")
    print(f"  Calories: {r['calories']}  Sugar: {r['sugar_g']}g  Caffeine: {r['caffeine_mg']}mg")
    print(f"  Price: ${r['price']}  Score: {r['score']}")
    print(f"  Desirability: {r['desirability']}  Affordability: {r['affordability']}  Seasonal: {r['seasonal']}")

results_df = pd.DataFrame(results)
display(results_df)

## Section 5 — Product Specifications & Revenue Impact

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# GENERATE PRODUCT SPEC CARDS
# ══════════════════════════════════════════════════════════════════════

# Focus on summer peak scenario (highest demand window)
summer = results_df[results_df['scenario'].str.contains('Summer Peak')].iloc[0]

print('=' * 60)
print('NEW FRAPPUCCINO PRODUCT SPECIFICATION')
print('Target Launch: Summer 2026')
print('=' * 60)
print(f"""\n
  Calories:    {summer['calories']} kcal
  Sugar:       {summer['sugar_g']}g
  Caffeine:    {summer['caffeine_mg']}mg
  Price:       ${summer['price']}
  
  Score:       {summer['score']} (max possible ~0.85)
  
  Scoring Breakdown:
    Desirability:  {summer['desirability']}
    Affordability: {summer['affordability']}
    Seasonal Fit:  {summer['seasonal']}
""")

# ── Revenue impact estimation ────────────────────────────────────────
# Based on synthetic data: avg Frappuccino transactions per month
monthly_frapp_txn = txn_frapp.groupby(txn_frapp['date'].dt.to_period('M')).size().mean()
print(f'Avg monthly Frappuccino transactions (synthetic): {monthly_frapp_txn:.0f}')

# If new product captures 15% of Frappuccino share (conservative for a new launch)
capture_rate = 0.15
new_product_monthly = monthly_frapp_txn * capture_rate
monthly_revenue = new_product_monthly * summer['price']
print(f'Projected monthly volume (15% capture): {new_product_monthly:.0f} transactions')
print(f'Projected monthly revenue: ${monthly_revenue:,.0f}')

# ── Compare with existing menu ───────────────────────────────────────
print('\n=== Comparison with Existing Frappuccinos ===')
print(f'{"Attribute":<15} {"New Product":<15} {"Menu Mean":<15} {"Menu Range"}')
for attr, col in [('Calories', 'calories'), ('Sugar (g)', 'sugar_g'), 
                   ('Caffeine (mg)', 'caffeine_mg'), ('Price ($)', 'price_usd')]:
    new_val = summer[col.replace('_usd', '')] if 'price' not in col else summer['price']
    m_mean = frapp_menu[col].mean()
    m_min = frapp_menu[col].min()
    m_max = frapp_menu[col].max()
    print(f'{attr:<15} {new_val:<15.1f} {m_mean:<15.1f} {m_min:.1f} - {m_max:.1f}')

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# SCENARIO COMPARISON VISUALIZATION
# ══════════════════════════════════════════════════════════════════════

fig = make_subplots(rows=1, cols=2, subplot_titles=('Optimal Price by Scenario', 'Score Breakdown'),
                    column_widths=[0.45, 0.55])

# Price comparison
fig.add_trace(go.Bar(
    y=results_df['scenario'], x=results_df['price'],
    orientation='h', marker_color='#00704A',
    text=[f'${p:.2f}' for p in results_df['price']],
    textposition='outside',
), row=1, col=1)

# Score breakdown stacked bar
for comp, color, name in [
    ('desirability', '#264653', 'Desirability'),
    ('affordability', '#E76F51', 'Affordability'),
    ('seasonal', '#2A9D8F', 'Seasonal Fit'),
]:
    fig.add_trace(go.Bar(
        y=results_df['scenario'], x=results_df[comp] * [0.35, 0.35, 0.35, 0.35, 0.35] if comp != 'seasonal' else results_df[comp] * 0.30,
        orientation='h', name=name, marker_color=color,
    ), row=1, col=2)

fig.update_layout(
    title='New Frappuccino: Optimization Across Market Scenarios',
    barmode='stack', height=450, template='plotly_white',
    legend=dict(orientation='h', yanchor='bottom', y=-0.2),
)
fig.update_xaxes(title_text='Price ($)', row=1, col=1)
fig.update_xaxes(title_text='Weighted Score Contribution', row=1, col=2)
fig.show()

## Section 6 — Sensitivity Analysis

How robust is the optimal design? We vary each parameter ±20% and observe score changes.

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# SENSITIVITY ANALYSIS
# ══════════════════════════════════════════════════════════════════════

base_params = {
    'calories': summer['calories'],
    'sugar_g': summer['sugar_g'],
    'caffeine_mg': summer['caffeine_mg'],
    'price': summer['price'],
}
base_score = summer['score']

fig_sens, axes = plt.subplots(2, 2, figsize=(12, 8))
param_list = list(base_params.items())

for idx, (param_name, base_val) in enumerate(param_list):
    ax = axes[idx // 2][idx % 2]
    
    # Vary ±30%
    pct_range = np.linspace(-0.30, 0.30, 30)
    scores = []
    for pct in pct_range:
        test_params = base_params.copy()
        test_params[param_name] = base_val * (1 + pct)
        s, _ = product_score(
            test_params['calories'], test_params['sugar_g'],
            test_params['caffeine_mg'], test_params['price'],
            temp_f=85, month=7, cpi=latest_cpi, wage=latest_wage
        )
        scores.append(s)
    
    ax.plot(pct_range * 100, scores, color='#00704A', linewidth=2)
    ax.axhline(base_score, color='red', linestyle='--', alpha=0.5, label=f'Base: {base_score:.4f}')
    ax.axvline(0, color='gray', linestyle=':', alpha=0.3)
    ax.set_title(f'Sensitivity: {param_name}')
    ax.set_xlabel('% Change from Optimal')
    ax.set_ylabel('Score')
    ax.legend(fontsize=8)
    
    # Score range
    score_range = max(scores) - min(scores)
    ax.text(0.95, 0.05, f'Range: {score_range:.4f}', transform=ax.transAxes,
            ha='right', fontsize=9, color='gray')

plt.suptitle('Sensitivity: How Robust is the Optimal Design?', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Rank parameters by sensitivity
print('\n=== Parameter Sensitivity Ranking ===')
sensitivities = []
for param_name, base_val in base_params.items():
    test_up = base_params.copy()
    test_up[param_name] = base_val * 1.10
    s_up, _ = product_score(test_up['calories'], test_up['sugar_g'], test_up['caffeine_mg'], test_up['price'],
                            85, 7, latest_cpi, latest_wage)
    test_dn = base_params.copy()
    test_dn[param_name] = base_val * 0.90
    s_dn, _ = product_score(test_dn['calories'], test_dn['sugar_g'], test_dn['caffeine_mg'], test_dn['price'],
                            85, 7, latest_cpi, latest_wage)
    sensitivities.append((param_name, abs(s_up - s_dn)))

for name, sens in sorted(sensitivities, key=lambda x: -x[1]):
    print(f'  {name:15s}: ±10% → Δscore = {sens:.4f}')

## Limitations

1. **Scoring function is hand-designed** — the weights (0.35/0.35/0.30) and Gaussian parameters are assumptions, not learned from real purchase data
2. **"Optimal" is relative to the scoring function** — different weight choices would produce different optimal products
3. **No competitive response** — assumes competitors don't react to the new product
4. **Revenue projection is illustrative** — the 15% capture rate is assumed, not modeled
5. **No taste/flavor modeling** — we optimize nutrition and price, not actual flavor appeal

The value is in the **framework**: given real POS data, the same optimization pipeline could be re-run with data-driven scoring weights.